# Ingesta de países

Lee `country.json` (JSON) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/countries` (modo `overwrite`). Para cada paso muestra varios métodos equivalentes con fines didácticos; prevalece el último que se ejecuta.

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

### Lectura del archivo

Primero lee `country.json` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
country_df = (
    spark.read
        .format('json')
        .option('inferSchema', True)
        .load(f'{bronze_folder_path}/country.json')
)
display(country_df)
country_df.printSchema()

In [0]:
country_schema = "countryId string, countryIsoCode string, countryName string"
country_df = (
    spark.read
        .format('json')
        .option('header', True)
        .schema(country_schema)
        .load(f'{bronze_folder_path}/country.json')
)
display(country_df)
country_df.printSchema()

### Eliminar columnas

#### Metodo 1

In [0]:
country_df_dropped = country_df.drop('countryIsoCode')

display(country_df_dropped)

#### Metodo 2

In [0]:
country_df_dropped = country_df.drop(country_df['countryIsoCode'])

display(country_df_dropped)

#### Metodo 3

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
country_df_dropped = country_df.drop(col('countryIsoCode'))

display(country_df_dropped)

### Renombrado y metadatos

Renombra `countryId` y `countryName` a snake_case y agrega `ingestion_date` y `env`.

In [0]:
country_df_final = (
    add_ingestion_date(country_df_dropped)
        .withColumnsRenamed({'countryId': 'country_id', 'countryName': 'country_name'})
        .withColumn('env', lit(v_env))
)
display(country_df_final)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/countries` en modo `overwrite`.

In [0]:
(
    country_df_final.write
        .mode('overwrite')
        .format('parquet')
        .save(f'{silver_folder_path}/countries')
)

### Verificación

Lista los archivos escritos en Silver.

In [0]:
display(dbutils.fs.ls(f'{silver_folder_path}/countries'))

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")